In [17]:
import torch
import torch.nn.functional as F
from torch import nn


In [18]:
#layer without parameters
class CenterLayer(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self,x):
        x-=x.mean()  # centered X
        return x
layer=CenterLayer()   # instantiate first, then give the real parameter
                      #when instantiate,parameter is needed as __init__set;   
layer(torch.FloatTensor([1,2,3,4,5]))   

tensor([-2., -1.,  0.,  1.,  2.])

In [19]:
#layer with parameters
class MyLinear(nn.Module):
    def __init__(self,in_units,units):    #in_units is input_dim or the input dim of MyLinear
        super().__init__()
        self.weight=nn.Parameter(torch.randn(in_units,units))   # weight param shape equals to the input_dim*output_dim 
        # self.bias=nn.Parameter(torch.randn(units,1))    XXXXXXXX  bias can not always be 1 ，up to batch_size
        self.bias=nn.Parameter(torch.randn(units,))          #bias parameter shape equals to the output_dim*?? （a column ）,a blank can auto-maych the dim
    def forward(self,x):
        return F.relu(torch.matmul(x,self.weight.data)+self.bias.data)
net=MyLinear(5,3)                     
net.weight,net.bias

#MICROSCOPE PERSPECTIVE:
#from the perspective of neural-network: this Linearlayer has 3 neurons,each can receive a 5_dim_tensor,while output a 1-dim-tensor
#all the three neurons can totally output a 3-dim-tensor
#bias is a inner part of a neuron (muliple-bias-output)


#MACRO PERSPECTIVE: a more important perspective
#this Linear Layer is a 5*3 matrix,the 5-dim-input will be processed to a 3-dim-output
#bias is after the procession upward,so the bias is 3-dim

'''
早期的教科书和课程往往从单个神经元开始讲起，因为这样便于理解“加权求和 → 加偏置 → 激活”的基本单元。
但到了现代深度学习框架（PyTorch、TensorFlow 等）中，实际开发时几乎总是以层（Layer）为单位来思考和构建网络。
原因有几个：
批量处理：框架一次性对整批数据做矩阵乘法，效率远高于逐个神经元循环。
自动微分：层级别的操作让计算图更简洁，反向传播也更高效。
模块化设计：nn.Linear 封装了权重矩阵和偏置向量，使用者只需指定输入/输出维度，无需手动管理每个神经元的参数。
硬件优化：GPU 擅长大规模矩阵运算，层视角正好匹配这种计算模式。
所以你的感受很自然：神经元视角是教学工具，层视角是工程实践。
在 PyTorch 中，nn.Linear 就是一个典型的层抽象——它的权重形状是 (out_features, in_features)，偏置形状是 (out_features,)，完全是从层的输入输出维度出发设计的。

'''


'\n早期的教科书和课程往往从单个神经元开始讲起，因为这样便于理解“加权求和 → 加偏置 → 激活”的基本单元。\n但到了现代深度学习框架（PyTorch、TensorFlow 等）中，实际开发时几乎总是以层（Layer）为单位来思考和构建网络。\n原因有几个：\n批量处理：框架一次性对整批数据做矩阵乘法，效率远高于逐个神经元循环。\n自动微分：层级别的操作让计算图更简洁，反向传播也更高效。\n模块化设计：nn.Linear 封装了权重矩阵和偏置向量，使用者只需指定输入/输出维度，无需手动管理每个神经元的参数。\n硬件优化：GPU 擅长大规模矩阵运算，层视角正好匹配这种计算模式。\n所以你的感受很自然：神经元视角是教学工具，层视角是工程实践。\n在 PyTorch 中，nn.Linear 就是一个典型的层抽象——它的权重形状是 (out_features, in_features)，偏置形状是 (out_features,)，完全是从层的输入输出维度出发设计的。\n\n'

layer can also act as part of net(nn.sequential)

In [20]:
net=nn.Sequential(nn.Linear(20,256),CenterLayer(),nn.ReLU(),nn.Linear(256,10),CenterLayer())
x=torch.rand(2,20)
y=net(x)
y.mean()   #so the one-dim-torch:y,could be nearly to 0

tensor(0., grad_fn=<MeanBackward0>)

layer can also be a part of Module

In [21]:
class CenterMod(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc=nn.Sequential(
        nn.Linear(20,256),
        CenterLayer(),
        nn.ReLU(),
        nn.Linear(256,10),
        CenterLayer()
            )
    def forward(self,x):
        return self.fc(x)
net=CenterMod()
y=net(x)
y.mean()

tensor(-1.6764e-09, grad_fn=<MeanBackward0>)

In [22]:
net=nn.Sequential(nn.Linear(20,256),MyLinear(256,128),nn.ReLU(),nn.Linear(128,64),nn.ReLU(),MyLinear(64,5))
x=torch.rand(2,20)
net(x)


tensor([[ 3.4168,  0.0000, 28.1454, 34.0610,  0.0000],
        [ 2.2383,  0.0000, 17.6985, 10.4532,  5.7729]],
       grad_fn=<ReluBackward0>)